In [ ]:
%load_ext autoreload
%autoreload 2
!pwd

import os
os.chdir("../../../")
!pwd

from plot_serializer.matplotlib.serializer import MatplotlibSerializer
from src.postprocessing.utils_plots import save_data_raw

In [ ]:
pt = 1./72.27
diss_tex_width = 390.0*pt

outdirectory = "plots/reevaluation/"

In [ ]:
general_information = {"author": {"name": "Julius H.P. Breuer", "ORCID": "0000-0002-6226-7208"}, "type": "PhD thesis, Technische Universität Darmstadt, Chair of Fluid Systems", "status": "in preparation", "title": "Algorithmische Systemplanung raumlufttechnischer Anlagen", "figure_created_with": "plot_reevaluation_many_load_cases_OFF.ipynb"}

def save_data(fig, serializer, outname, filedata, caption):
    save_data_raw(fig, serializer, outdirectory, outname, filedata, caption, general_information=general_information, id_result_subfolder = "id_results/")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("FST_bw.mplstyle")
from src.postprocessing.utils_plots import create_standard_list

from matplotlib.font_manager import FontProperties
arial_font = FontProperties(family="Arial", style="italic")

save_plots = input("Save plots?")
if save_plots == "y":
    save_plots = True
else:
    save_plots = False

## Plot

In [ ]:
def get_filenames_optimisation_type(folder_path, cs_names, optimisation_type="Topology"):
    paths = [folder_path + cs + "/" for cs in cs_names]

    standard_dict_list = create_standard_list(paths, optimisation_type, False)
    
    files = []
    for cs, x in zip(cs_names, standard_dict_list):
        if len(x) == 1:
            files.append()
        else:
            files.extend([folder_path + cs + "/" + str(y) for y in x["filename"]])
    
    return files

In [ ]:
folder_path1 = "results/off/combined/max_velocity_5_10ms_max_height_04m/Monte_Carlo_Validation_acoustics/"

cs_names_all = ["VAV-CPC", "VAV-VPC", "DF-CPC"] #,  "DF-CPC"]
# cs_names_all = ["VAV-VPC"]

markerstyle = ["o","D", "s","v","^","P"]

ticknames_all = ["KONST/\nZENTRAL", "VAR/\nZENTRAL", "KONST/\nVERTEILT"]
ticknames_min_noise = [ticknames_all[1], ticknames_all[-1]]

# ticknames_all = ["VAR/\nZENTRAL"]*2

files1 = get_filenames_optimisation_type(folder_path1, cs_names_all, "Configuration")
# files2 = get_filenames_optimisation_type(folder_path2, ["VAV-VPC"], "Configuration")

# files1 = [files1[0], *files1[3:]]

del files1[2]

In [ ]:
import h5py
import pandas as pd

sampled_power_consumption = f"/Postprocessing/Multi Load Case Comparison/Points {128}"   # change to your dataset path

fanresults = "/Postprocessing/FanResults"

# exclude_cols = ['time share', 'power consumption in W', 'power consumption lower bound in W']

exclude_cols = ['sound pressure gap in dB', 'time share', 'power consumption in W', 'power consumption lower bound in W']



def load_optimisation_results(path):
    print(path)
    with h5py.File(path, "r") as f:

        scenario = f["Optimisation Components/Set/Scenarios"][:].astype(int)

        volume_flow = {int(s): float() for s in scenario}
        for s in scenario:
            max_volume_flow = np.max(np.array(f[f"Optimisation Components/Parameter/Scenario/{s}/volume_flow"]["value"]))
            volume_flow[s] = max_volume_flow

        # compute optimisation problems power consumption and get time share for comparison

        fan_power = pd.DataFrame(f[fanresults][...])
        fan_power["Scenario"] = fan_power["Scenario"].astype(int)
        fan_power = fan_power.groupby('Scenario')[['PowerConsumption', 'VolumeFlow']].sum().reset_index()
        fan_power = fan_power.set_index("Scenario")

        fan_power.index.name = "Scenarios"


        time_share = f["Optimisation Components/Parameter/time_share"][...]
        time_share = pd.DataFrame(time_share, columns=['Scenarios', 'value'])
        time_share['Scenarios'] = time_share['Scenarios'].astype(int) # time_share['Scenarios'] = time_share['Scenarios'].astype('U').str.strip().replace('', np.nan).astype(float).astype(int)
        time_share['value'] = time_share['value'].astype(float)
        time_share = time_share[['Scenarios', 'value']]
        time_shares = time_share.set_index("Scenarios")["value"]

        power_consumption = fan_power.join(time_shares.rename('TimeShare'))

        power_consumption["VolumeFlow"] = power_consumption.index.map(volume_flow)

    return power_consumption


def load_postprocessing_results(path):
    print(path)
    with h5py.File(path, "r") as f:
        dset = f[sampled_power_consumption]
        arr = dset[...]   # or dset[:]
        df = pd.DataFrame.from_records(arr)
       
        df["total volume flow in m³/h"] = df.drop(columns=exclude_cols).sum(axis=1)
    return df

In [ ]:
dfs,dfs2 = [],[]


for path in files1:
    df = load_postprocessing_results(path)
    dfs.append(df)

    df2 = load_optimisation_results(path)
    dfs2.append(df2)

    # opt_fan_powers.append(opt_fan_power)

In [ ]:
for df,df2 in zip(dfs,dfs2):
    print(f'Max Power consumption: {df["power consumption in W"].max()}, Average: {sum(df["power consumption in W"] * df["time share"])}')
    print(f'Max Power consumption: {df2["PowerConsumption"].max()}, Average: {sum(df2["PowerConsumption"] * df2["TimeShare"])}')

post_power_consumption = [sum(x["power consumption in W"] * x["time share"]) for x in dfs]
opt_power_consumption = [sum(x["PowerConsumption"] * x["TimeShare"]) for x in dfs2]

x = np.arange(0,len(dfs))


serializer = MatplotlibSerializer()
fig,ax = serializer.subplots(figsize=(0.8*diss_tex_width,diss_tex_width*5/9))

ax.bar(x-0.2, post_power_consumption, width=0.4, label="Monte-Carlo", color="black")
ax.bar(x+0.2, opt_power_consumption, width=0.4, label="Norm", color="grey")
_ = ax.set_xticks(x, ticknames_all)
ax.set_ylabel("MITTLERE LEISTUNG in W")

ax.legend(
    loc="upper right", prop=arial_font
    # prop={'weight': 'bold'}
)

In [ ]:
np.array(post_power_consumption) / np.array(opt_power_consumption)

In [ ]:
for df in dfs:
    print(len(df.loc[df["power consumption in W"]<0]) / len(df))

In [ ]:
for i in range(len(dfs)):
    ratio = opt_power_consumption[i] / post_power_consumption[i] - 1

    print(f"Optimierung überschätzt Leistungsaufnahme um {ratio*100:.0f} %")

In [ ]:
serializer = MatplotlibSerializer()

fig, ax = serializer.subplots(1,len(dfs),figsize=(diss_tex_width,diss_tex_width*5/9), sharey=True)
# axs = axs.flatten()
# ax = [ax]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

markerstyle_curr = [markerstyle[1], markerstyle[2], markerstyle[3]]

for i in range(len(dfs)):
    color = "black" #colors[i % len(colors)]
    # ax = axs[i]
    # plt.scatter(dfs[i]["total volume flow in m³/h"], dfs[i]["power consumption in W"], s=dfs[i]["time share"]*1e1, color=color, label=names[i], zorder=i)

    ax[i].scatter(dfs[i]["total volume flow in m³/h"], dfs[i]["power consumption in W"], s=7, facecolor="white", edgecolor=color, linewidth=0.5, marker=markerstyle_curr[i], label=ticknames_all[i], zorder=i)

    ax[i].scatter(dfs2[i]["VolumeFlow"]*3600, dfs2[i]["PowerConsumption"],marker=markerstyle_curr[i], edgecolor = color, facecolor="grey", s= 40, zorder=i)

    ax[i].set_title(ticknames_all[i], fontsize=10)
    # ax[i].set_xlabel("VOLUMENSTROM in m³/h")
    ax[i].set_xlim(right=22500)
    ax[i].set_ylim(bottom=0, top=5500)

ax[0].set_ylabel("LEISTUNGSAUFNAHME in W")
fig.supxlabel("VOLUMENSTROM in m³/h", y=-0.05, fontsize=10)
# ax[0].legend(markerscale=2)

In [ ]:
fig, ax = plt.subplots(len(dfs),1, figsize=(3,3), sharey=True, sharex=True)

if not isinstance(ax, np.ndarray):
    ax = [ax]

for i in range(len(dfs)):
    ax[i].hist(dfs[i]["power consumption in W"], bins=30, weights=dfs[i]["time share"], edgecolor="black")
    # ax[i].bar(real_power_consumption,real_time_share,width=30,color="blue", zorder=-1)
    ax[i].set_ylabel(ticknames_all[i])

    ax[i].bar(dfs2[i]["PowerConsumption"],dfs2[i]["TimeShare"]+0.01, color="red", width=40, zorder=1)

In [ ]:
serializer = MatplotlibSerializer()
fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))
colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff']

markerstyle_curr = [markerstyle[1], markerstyle[2], markerstyle[3]]

for i in range(len(dfs)):
    # if i == 0:
    #     continue
    color = colors[i % len(colors)]

    ax.scatter(dfs[i]["total volume flow in m³/h"], dfs[i]["sound pressure gap in dB"], s=7, marker= markerstyle_curr[i], facecolors="white", linewidths=0.5, edgecolors=color, zorder=-i, label=ticknames_all[i])

    # ax.scatter()

    ax.set_xlabel("VOLUMENSTROM in m³/h")

    ax.set_ylabel("PEGELABWEICHUNG in dB")
    # break

ax.axhline(0, zorder=-3)
ax.text(1700,1.2,"unzulässig",va="bottom", fontfamily='Arial', fontstyle='italic')
ax.text(1700,-1.2,"zulässig",va="top", fontfamily='Arial', fontstyle='italic')



for ii in range(3):
    ax.plot([1700+400*ii,2200+400*ii],[0,0.5],"black")

ax.legend(loc="lower right", handletextpad=-0.1, markerscale=1.5)


In [ ]:
dfs[0].loc[dfs[0]["sound pressure gap in dB"] > 0, "sound pressure gap in dB"].max()

In [ ]:
for df in dfs:
    print(f"Percent of load cases exceeding the sound pressure limit {sum(df['sound pressure gap in dB']>0) / len(df)}")

## K-Medoid Ergebnisvergleich

In [ ]:
folder_path1 = "results/off/combined/max_velocity_5_10ms_max_height_04m/Monte_Carlo_Validation_acoustics/"

cs_names_all = ["VAV-CPC", "VAV-VPC", "DF-CPC"] #,  "DF-CPC"]
cs_names_all = ["VAV-VPC"]

markerstyle = ["o","D", "s","v","^","P"]

ticknames_all = ["KONST/\nZENTRAL", "VAR/\nZENTRAL", "KONST/\nVERTEILT"]
ticknames_min_noise = [ticknames_all[1], ticknames_all[-1]]

ticknames_all = ["NORM", "K-MEDOID"]

files1 = get_filenames_optimisation_type(folder_path1, cs_names_all, "Configuration")

In [ ]:
dfs,dfs2 = [],[]

for path in files1:
    df = load_postprocessing_results(path)
    dfs.append(df)

    df2 = load_optimisation_results(path)
    dfs2.append(df2)

    # opt_fan_powers.append(opt_fan_power)

In [ ]:
serializer = MatplotlibSerializer()

fig, ax = serializer.subplots(1,len(dfs),figsize=(diss_tex_width,diss_tex_width*5/9), sharey=True)
# axs = axs.flatten()
# ax = [ax]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

markerstyle_curr = [markerstyle[1], markerstyle[2], markerstyle[3]]

for i in range(len(dfs)):
    color = "black" #colors[i % len(colors)]
    # ax = axs[i]
    # plt.scatter(dfs[i]["total volume flow in m³/h"], dfs[i]["power consumption in W"], s=dfs[i]["time share"]*1e1, color=color, label=names[i], zorder=i)

    ax[i].scatter(dfs[i]["total volume flow in m³/h"], dfs[i]["power consumption in W"], s=7, facecolor="white", edgecolor=color, linewidth=0.5, marker=markerstyle_curr[i], label=ticknames_all[i], zorder=i)

    ax[i].scatter(dfs2[i]["VolumeFlow"]*3600, dfs2[i]["PowerConsumption"],marker=markerstyle_curr[i], edgecolor = color, facecolor="grey", s= 40, zorder=i)

    ax[i].set_title(ticknames_all[i], fontsize=10)
    # ax[i].set_xlabel("VOLUMENSTROM in m³/h")
    ax[i].set_xlim(right=22500)
    ax[i].set_ylim(bottom=0, top=5500)

ax[0].set_ylabel("LEISTUNGSAUFNAHME in W")
fig.supxlabel("VOLUMENSTROM in m³/h", y=-0.05, fontsize=10)
# ax[0].legend(markerscale=2)

fig.tight_layout()

if save_plots:
    outname = "monte_carlo_power_consumption_norm_vs_kmedoid_off"
    filedata = [file.split("/")[-1] for file in files1]
    caption = "Leistungsaufnahme der Regelstrategie \gls{vavvpc} des Büroturms über dem Gesamtvolumenstrom für alle Monte-Carlo-Lastfälle. Verglichen werden die Ergebnisse der Optimierung auf Basis der Normlastfälle (links) und der mittels k-Medoid-Clustering bestimmten Lastfälle (rechts). Hervorgehobene Punkte kennzeichnen die jeweils in der Optimierung berücksichtigten Lastfälle. Die Optimierung auf Basis der Normlastfälle repräsentiert die Monte-Carlo-Lastfälle nur unzureichend, wodurch die mittlere Leistungsaufnahme um \SI{13}{\percent} unterschätzt wird. Die k-Medoid-Lastfälle bilden die Monte-Carlo-Lastfälle deutlich besser ab und reduzieren die Unterschätzung der mittleren Leistungsaufnahme auf \SI{5}{\percent}."
    save_data(fig, serializer, outname, filedata, caption)

In [ ]:
serializer = MatplotlibSerializer()
fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))
colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff']

markerstyle_curr = [markerstyle[1], markerstyle[2], markerstyle[3]]

for i in range(len(dfs)):
    # if i == 0:
    #     continue
    color = colors[i % len(colors)]

    ax.scatter(dfs[i]["total volume flow in m³/h"], dfs[i]["sound pressure gap in dB"], s=7, marker= markerstyle_curr[i], facecolors="white", linewidths=0.5, edgecolors=color, zorder=-i, label=ticknames_all[i])

    # ax.scatter()

    ax.set_xlabel("VOLUMENSTROM in m³/h")

    ax.set_ylabel("PEGELABWEICHUNG in dB")
    # break

ax.axhline(0, zorder=-3)
ax.text(1700,1.2,"UNZULÄSSIG",va="bottom", fontfamily='Arial', fontstyle='italic')
ax.text(1700,-1.2,"ZULÄSSIG",va="top", fontfamily='Arial', fontstyle='italic')



for ii in range(3):
    ax.plot([1700+400*ii,2200+400*ii],[0,0.5],"black")

ax.legend(loc="lower right", handletextpad=-0.1, markerscale=1.5)